In [ ]:
# Imports and initial setup
import os
import numpy as np
import seaborn as sns
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, GlobalAveragePooling2D, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix

# Enable optimizations
tf.keras.mixed_precision.set_global_policy('mixed_float16')
tf.config.optimizer.set_jit(True)

# Initialize strategy
strategy = tf.distribute.MirroredStrategy()
print(f'Number of devices: {strategy.num_replicas_in_sync}')

In [ ]:
# Hyperparameters
IMG_SIZE = (256, 256)
BASE_BATCH_SIZE = 64
BATCH_SIZE = BASE_BATCH_SIZE * strategy.num_replicas_in_sync
EPOCHS = 20

# Dataset setup
data_dir = 'data'
train_dir = os.path.join(data_dir, 'train')
val_dir = os.path.join(data_dir, 'val')
test_dir = os.path.join(data_dir, 'test')

# Data pipeline
AUTOTUNE = tf.data.experimental.AUTOTUNE

def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = image / 255.0
    return image, label

def configure_dataset(ds):
    ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE)
    ds = ds.cache()
    ds = ds.shuffle(1000)
    return ds.prefetch(buffer_size=AUTOTUNE)

train_data = configure_dataset(tf.keras.preprocessing.image_dataset_from_directory(
    train_dir, batch_size=BATCH_SIZE, image_size=IMG_SIZE, label_mode='categorical'))

val_data = configure_dataset(tf.keras.preprocessing.image_dataset_from_directory(
    val_dir, batch_size=BATCH_SIZE, image_size=IMG_SIZE, label_mode='categorical'))

test_data = configure_dataset(tf.keras.preprocessing.image_dataset_from_directory(
    test_dir, batch_size=BATCH_SIZE, image_size=IMG_SIZE, label_mode='categorical'))

In [ ]:
# CNN Model Definition
with strategy.scope():
    model = Sequential([
        tf.keras.layers.Input(shape=(*IMG_SIZE, 3)),
        Conv2D(16, 3, activation='relu'),
        BatchNormalization(),
        MaxPooling2D(2),
        
        Conv2D(32, 3, activation='relu'),
        BatchNormalization(),
        MaxPooling2D(2),
        
        Conv2D(64, 3, activation='relu'),
        BatchNormalization(),
        MaxPooling2D(2),
        
        Conv2D(128, 3, activation='relu'),
        BatchNormalization(),
        MaxPooling2D(2),
        
        GlobalAveragePooling2D(),
        Dense(128, activation='relu'),
        Dense(10, activation='softmax')
    ])
    
    model.compile(optimizer=tf.keras.optimizers.Adam(0.001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    model.summary()

In [ ]:
# Training
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS,
    callbacks=[
        EarlyStopping(patience=10, restore_best_weights=True),
        ModelCheckpoint('best_model.keras', save_best_only=True)
    ]
)

In [ ]:
# Save model
model.save('plant_disease_cnn.keras')